Learn how to fine-tune a pretrained DistilBERT model for binary sentiment classification using Hugging Face Transformers and PyTorch. The notebook covers data preparation, tokenization, model training, evaluation, and inference in a complete end-to-end NLP workflow.

In [1]:
#============================================================================
# 0. INSTALLS AND IMPORTS
#============================================================================
# If you are using Google Colab, uncomment the next line:
# !pip install -q transformers scikit-learn

import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

#============================================================================
# 1. SETUP AND CONFIGURATION
#============================================================================

def set_seed(seed=42):
    """Sets random seeds for reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# A lightweight BERT-style model already fine-tuned on binary sentiment.
# This keeps the example fast and beginner-friendly.
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

NUM_LABELS = 2  # 0 = Negative, 1 = Positive
MAX_LEN = 128
BATCH_SIZE = 4
LEARNING_RATE = 2e-5
NUM_EPOCHS = 2

label_names = {
    0: "Negative",
    1: "Positive"
}

#============================================================================
# 2. DATA PREPARATION
#============================================================================
# Small book-review-style dataset for demonstration.
# In a real project, you would load hundreds or thousands of reviews from a CSV.

reviews = [
    "This book was excellent and very easy to understand.",
    "I loved the examples and the explanations were clear.",
    "A wonderful introduction with practical code.",
    "This was helpful, engaging, and well organized.",
    "The book made difficult ideas feel simple.",
    "I enjoyed every chapter and learned a lot.",
    "The writing was clear and the projects were useful.",
    "A fantastic learning resource for beginners.",

    "This book was confusing and poorly organized.",
    "I did not enjoy the explanations at all.",
    "The examples were unclear and frustrating.",
    "This was boring and hard to follow.",
    "The book felt rushed and incomplete.",
    "I would not recommend this to beginners.",
    "The chapters were difficult to understand.",
    "The explanations left me more confused than before."
]

labels = [
    1, 1, 1, 1, 1, 1, 1, 1,   # Positive
    0, 0, 0, 0, 0, 0, 0, 0    # Negative
]

train_reviews, val_reviews, train_labels, val_labels = train_test_split(
    reviews,
    labels,
    test_size=0.25,
    random_state=42,
    stratify=labels
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#============================================================================
# 3. CUSTOM PYTORCH DATASET
#============================================================================

class BookReviewDataset(Dataset):
    """Custom Dataset for processing book review text."""

    def __init__(self, reviews, labels, tokenizer, max_len):
        self.reviews = reviews
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        """Returns the number of samples in the dataset."""
        return len(self.reviews)

    def __getitem__(self, item_index):
        """
        Retrieves and preprocesses a single review.
        This is where tokenization happens.
        """
        review = str(self.reviews[item_index])
        label = int(self.labels[item_index])

        encoding = self.tokenizer(
            review,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_dataset = BookReviewDataset(train_reviews, train_labels, tokenizer, MAX_LEN)
val_dataset = BookReviewDataset(val_reviews, val_labels, tokenizer, MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

#============================================================================
# 4. MODEL, OPTIMIZER, AND LOSS
#============================================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# The loss function is handled internally when labels are provided.

#============================================================================
# 5. TRAINING AND EVALUATION FUNCTIONS
#============================================================================

def train_epoch(model, data_loader, optimizer, device):
    """Trains the model for one full pass over the training data."""
    model.train()
    total_loss = 0

    for batch in data_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    return total_loss / len(data_loader)


def evaluate_model(model, data_loader, device):
    """Evaluates the model on validation data."""
    model.eval()

    total_loss = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            predictions = torch.argmax(logits, dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions)

    return total_loss / len(data_loader), accuracy, f1

#============================================================================
# 6. MAIN TRAINING LOOP
#============================================================================

print("Starting model training...")

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_dataloader, optimizer, DEVICE)
    val_loss, val_accuracy, val_f1 = evaluate_model(model, val_dataloader, DEVICE)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

print("Training complete!")

#============================================================================
# 7. PREDICTION (INFERENCE)
#============================================================================

def predict_sentiment(text, model, tokenizer, device, max_len):
    """Predicts sentiment for a single piece of text."""
    model.eval()

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

    return label_names[predicted_class]

#============================================================================
# 8. TEST THE FINAL MODEL
#============================================================================

test_reviews = [
    "I absolutely loved this book, a true masterpiece.",
    "This was the most boring book I have ever read.",
    "The plot was a bit slow, but I still enjoyed it.",
    "I'm not sure what to think about this one."
]

print("\n--- Testing the final model ---")

for review in test_reviews:
    sentiment = predict_sentiment(review, model, tokenizer, DEVICE, MAX_LEN)
    print(f"Review: {review}")
    print(f"Sentiment: {sentiment}")
    print("-" * 50)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Starting model training...
Epoch 1/2 | Train Loss: 0.2239 | Val Loss: 0.0002 | Val Accuracy: 1.0000 | Val F1: 1.0000
Epoch 2/2 | Train Loss: 0.0074 | Val Loss: 0.0003 | Val Accuracy: 1.0000 | Val F1: 1.0000
Training complete!

--- Testing the final model ---
Review: I absolutely loved this book, a true masterpiece.
Sentiment: Positive
--------------------------------------------------
Review: This was the most boring book I have ever read.
Sentiment: Negative
--------------------------------------------------
Review: The plot was a bit slow, but I still enjoyed it.
Sentiment: Positive
--------------------------------------------------
Review: I'm not sure what to think about this one.
Sentiment: Negative
--------------------------------------------------
